## plan
use SAGE modelling method (t-SNE into DBSCAN)

In [ ]:
from sklearn.manifold import TSNE
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

In [ ]:
## CONSTANTS

data_path = './data/ACOD_CTD_v1.0.nc'

### data cleaning (testing)

In [ ]:
def null_frac(da):

    return da.isnull().sum().data / da.size

In [ ]:
full_data = xr.open_dataset('./data/ACOD_CTD_v1.0.nc')

In [ ]:
top_layer = full_data.sel(PRESSURE=slice(0,50))

print(null_frac(full_data.Temperature))
print(null_frac(top_layer.Temperature))

In [ ]:
top_temp = top_layer.Temperature
itp_temp = top_layer.Temperature.interpolate_na(dim='PRESSURE')
null_frac(itp_temp)

In [ ]:
remasked = top_temp.where(~top_temp.sel(PRESSURE=50).isnull())
itp_remasked = remasked.interpolate_na(dim='PRESSURE')

null_frac(itp_remasked)

In [ ]:
depths = list(range(10, 1000, 10))

null_fracs = []
for i in depths:
    top_layer = full_data.Temperature.sel(PRESSURE=slice(0,i))
    null_fracs.append(null_frac(top_layer))

plt.plot(depths, null_fracs)

In [ ]:
depths = list(range(10, 500, 10))

null_fracs = []
itp_nf = []
for i in depths:
    top_layer = full_data.Temperature.sel(PRESSURE=slice(0,i))
    itp_tl = top_layer.interpolate_na(dim='PRESSURE')
    null_fracs.append(null_frac(top_layer))
    itp_nf.append(null_frac(itp_tl))

import matplotlib.pyplot as plt

plt.plot(depths, null_fracs, label='original')
plt.plot(depths, itp_nf, label='interpolated')

plt.legend()

In [ ]:
depths = list(range(10, 80))

null_fracs = []
itp_nf = []
for i in depths:
    top_layer = full_data.Temperature.sel(PRESSURE=slice(0,i))
    itp_tl = top_layer.interpolate_na(dim='PRESSURE')
    null_fracs.append(null_frac(top_layer))
    itp_nf.append(null_frac(itp_tl))

import matplotlib.pyplot as plt

plt.plot(depths, null_fracs, label='original')
plt.plot(depths, itp_nf, label='interpolated')

plt.legend()

### data cleaning (proper)

In [ ]:
data = xr.open_dataset(data_path)
upper_data = data.sel(PRESSURE=slice(0, 50))

tot_profiles = len(upper_data.PROFILE)

data

In [ ]:
temp_array = upper_data.interpolate_na(dim='PRESSURE').Temperature.data

'''for i in range(tot_profiles):
    print(np.sum(np.isnan(temp_array[:, i])))'''

nan_mask = ~np.sum(np.isnan(temp_array),axis=0).astype(bool)
# originally 29717 values; ended up with 23604
clean_temperature = temp_array[:,nan_mask]

In [ ]:
data.sel(PROFILE=nan_mask)

### code

In [ ]:
t_SNE_embedding = TSNE(n_components=3, learning_rate='auto', init='random', perplexity=3)

In [ ]:
temp_embedded = t_SNE_embedding.fit_transform(clean_temperature)

In [ ]:
fig, ax = plt.subplots(subplot_kw={'projection': '3d'})


x, y, z = temp_embedded.T
ax.scatter(x, y, z)